# Tier 3: Molecular Modeling & Force Fields
## Lab 2.0 — Molecular Mechanics, Force Fields, and Energy Minimization

Welcome to **Lab 2.0** of the *Computational Structural Biology and Generative Macromolecular Design* curriculum!

### Why This Lab Matters
In Tier 1 and Tier 2, we examined experimental coordinate records (PDB files) and inferred ancestral sequences from evolutionary alignments. However, static experimental models and raw designed sequences are rarely physically relaxed:
1. **Crystallographic Artifacts:** Coordinates refined against medium-resolution diffraction maps frequently contain **steric clashes** (atoms placed closer than the sum of their van der Waals radii) or strained bond angles due to lattice contacts.
2. **Artificial Unfavorable Energetics:** AI models (such as AlphaFold, ESMFold, and diffusion architectures) output backbone and side-chain geometries that can contain minor steric overlaps or unrelaxed peptide bonds that would instantly blow up an atomistic molecular dynamics simulation.
3. **The Physics Engine of Structural Biology:** Molecular mechanics (MM) replaces quantum mechanical electronic wavefunctions with **empirical classical potential energy functions** parameterized against quantum chemistry and physical experiments (e.g., Amber, CHARMM, OPLS, GROMOS).

Before running microsecond-scale molecular dynamics or simulating protein-ligand binding, every computational structural biologist must know how to:
- Decompose the molecular mechanics potential energy function into bonded ($V_{\text{bond}}, V_{\text{angle}}, V_{\text{dihedral}}$) and non-bonded ($V_{\text{vdW}}, V_{\text{elec}}$) contributions.
- Compute analytical negative energy gradients ($-\nabla V$) to determine the instantaneous forces acting on atoms.
- Understand the mathematical differences between **Steepest Descent** (for relief of severe steric overlaps) and **L-BFGS / Conjugate Gradient** (quasi-Newton optimization for rapid convergence to local minima).
- Set up an atomistic force field (Amber14-SB) in **OpenMM** to relax real biological structures.

---

### Learning Objectives
By the end of this laboratory, you will be able to:
- Formulate the classical Class I potential energy function and calculate individual energy terms for bonded and non-bonded atom pairs.
- Implement a pure-Python 1D and 3D Steepest Descent energy minimizer resolving Lennard-Jones steric clashes.
- Contrast the convergence properties of Steepest Descent versus Conjugate Gradient / L-BFGS algorithms.
- Build an OpenMM simulation topology, parameterize *Ideonella sakaiensis* PETase (PDB: `6ANE`) with the **Amber14-SB** force field, and run energy minimization in vacuum and implicit solvent.
- Quantify structural relaxation by computing per-residue root-mean-square displacement vectors ($\Delta \mathbf{r}$) and visualizing relaxed active-site geometries in `py3Dmol`.
- Self-assess your conceptual understanding using embedded `jupyterquiz` formative diagnostic questions.

---

### Running Systems in Tier 3
- **Protein System:** *Ideonella sakaiensis* **PETase** (PDB: `6ANE`), focusing on relieving steric strain in the catalytic triad ($	ext{Ser160}$–$	ext{Asp206}$–$	ext{His237}$) and the dynamic $	ext{Trp185}$ aromatic clamp.
- **Nucleic Acid System:** *Vibrio vulnificus* *add* **A-riboswitch** (PDB: `1Y26`), exploring high-density electrostatic repulsion in polyanionic phosphate backbones.

Let's begin by preparing our Google Colab computational environment.


### Section 0: Environment Setup and Pinned Dependencies
We install OpenMM (the high-performance GPU/CPU molecular simulation toolkit) alongside our core analysis and visualization packages.


In [ ]:
# Install OpenMM, PDBFixer, BioPython, py3Dmol, and jupyterquiz
!pip install -q openmm>=8.2.0 pdbfixer biopython==1.83 jupyterquiz==2.8.1 py3Dmol==2.1.0 numpy pandas scipy matplotlib requests

import os
import sys
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol
from Bio.PDB import PDBParser, PDBIO, Select

# OpenMM & PDBFixer imports
import openmm as mm
from openmm import app
from openmm import unit
from pdbfixer import PDBFixer
from IPython.display import display

# Pinned course reproducibility seed
COURSE_SEED = 42
random.seed(COURSE_SEED)
np.random.seed(COURSE_SEED)

os.makedirs("data", exist_ok=True)
os.makedirs("manifests", exist_ok=True)

print("Lab 2.0 environment initialized successfully.")
print(f"Python version: {sys.version.split()[0]}")
print(f"OpenMM version: {mm.__version__} | BioPython version: 1.83")
print(f"Reproducibility seed: {COURSE_SEED}")


---
## Section 1: Anatomy of a Classical Class I Force Field

In molecular mechanics, atoms are treated as classical point masses with fixed partial charges, connected by springs. The total potential energy $V(\mathbf{R})$ of a conformation $\mathbf{R} = (\mathbf{r}_1, \mathbf{r}_2, \dots, \mathbf{r}_N)$ is decomposed into **bonded** and **non-bonded** terms:

$$V_{\text{total}}(\mathbf{R}) = V_{\text{bond}} + V_{\text{angle}} + V_{\text{dihedral}} + V_{\text{vdW}} + V_{\text{elec}}$$

### 1. Bond Stretching (Harmonic Approximation)
Covalent bonds are modeled as harmonic springs around an equilibrium bond length $r_0$:
$$V_{\text{bond}} = \sum_{\text{bonds}} k_b (r - r_0)^2$$
- $k_b$: Bond force constant (typically $\approx 300 - 500\,\text{kcal}/(\text{mol}\cdot\text{\AA}^2)$).
- $r_0$: Equilibrium bond distance (e.g., $\text{C}_\alpha - \text{C} \approx 1.52\,\text{\AA}$).
- *Limitation:* Harmonic springs cannot model chemical bond breaking or radical reactions.

### 2. Angle Bending (Harmonic Approximation)
Bond angles formed by three covalently bonded atoms ($A-B-C$) are modeled around equilibrium angle $\theta_0$:
$$V_{\text{angle}} = \sum_{\text{angles}} k_\theta (\theta - \theta_0)^2$$
- $k_\theta$: Angle force constant (typically $\approx 50 - 100\,\text{kcal}/(\text{mol}\cdot\text{rad}^2)$).
- $\theta_0$: Equilibrium angle (e.g., tetrahedral $sp^3 \approx 109.5^{\circ}$, planar $sp^2 \approx 120.0^{\circ}$).

### 3. Torsional Dihedrals (Fourier Series)
Rotation around four sequentially bonded atoms ($A-B-C-D$) across dihedral angle $\phi$:
$$V_{\text{dihedral}} = \sum_{\text{dihedrals}} \sum_{n} \frac{V_n}{2} \left[ 1 + \cos(n \phi - \gamma) \right]$$
- $n$: Periodicity (e.g., $n = 3$ for threefold staggered rotamers in $sp^3-sp^3$ alkane bonds; $n = 2$ for planar peptide $\omega$ bonds).
- $\gamma$: Phase offset angle.
- $V_n$: Torsional barrier height.

### 4. van der Waals Interactions (Lennard-Jones 12-6 Potential)
Non-bonded steric repulsion and London dispersion attraction between non-covalently linked atoms separated by distance $r_{ij}$:
$$V_{\text{vdW}} = \sum_{i < j} 4 \epsilon_{ij} \left[ \left( \frac{\sigma_{ij}}{r_{ij}} \right)^{12} - \left( \frac{\sigma_{ij}}{r_{ij}} \right)^6 \right]$$
- $\sigma_{ij}$: Distance at which the potential is zero (related to van der Waals radius: $r_{\min} = 2^{1/6} \sigma \approx 1.122 \sigma$).
- $\epsilon_{ij}$: Potential well depth (dispersion attraction minimum).
- **The $r^{-12}$ Repulsive Wall:** Rapidly escalates to $+\infty$ when atoms overlap ($r_{ij} < \sigma$), preventing atomic interpenetration.
- **The $r^{-6}$ Attractive Well:** Models induced dipole-induced dipole London dispersion.

### 5. Electrostatic Interactions (Coulomb's Law)
Point-charge interactions between partial atomic charges $q_i$ and $q_j$:
$$V_{\text{elec}} = \sum_{i < j} \frac{q_i q_j}{4 \pi \epsilon_0 \epsilon_r r_{ij}}$$
- $q_i, q_j$: Fixed partial atomic charges assigned by the force field (e.g., Amber RESP charges).
- $\epsilon_r$: Relative dielectric permittivity ($\epsilon_r = 1$ in explicit solvent; $\epsilon_r \approx 4 - 80$ in distance-dependent or implicit solvent models).

Let's implement a pure-Python script evaluating Lennard-Jones and Coulomb potentials across an interatomic distance track.


In [ ]:
# Pure-Python potential energy calculator
def lennard_jones_energy(r: np.ndarray, sigma: float = 3.4, epsilon: float = 0.1) -> np.ndarray:
    """Calculates standard 12-6 Lennard-Jones potential in kcal/mol (r in Å)."""
    # Avoid division by zero
    r_safe = np.maximum(r, 0.1)
    ratio = sigma / r_safe
    return 4.0 * epsilon * (ratio**12 - ratio**6)

def coulomb_energy(r: np.ndarray, q1: float, q2: float, epsilon_r: float = 1.0) -> np.ndarray:
    """
    Calculates Coulomb electrostatic potential in kcal/mol (r in Å).
    Coulomb conversion constant: e^2 / (4 * pi * eps0 * 1 Å) = 332.0637 kcal*Å / (mol*e^2)
    """
    r_safe = np.maximum(r, 0.1)
    COULOMB_CONST = 332.0637
    return (COULOMB_CONST * q1 * q2) / (epsilon_r * r_safe)

# Distance sweep from 1.5 Å (severe clash) to 8.0 Å
r_dist = np.linspace(1.8, 8.0, 300)

# Typical parameters: Carbon-Carbon van der Waals (sigma = 3.4 Å, eps = 0.086 kcal/mol)
v_lj = lennard_jones_energy(r_dist, sigma=3.4, epsilon=0.086)

# Electrostatics: Salt bridge (q1 = +0.5 e, q2 = -0.5 e, eps_r = 1.0)
v_elec_opp = coulomb_energy(r_dist, q1=0.5, q2=-0.5, epsilon_r=1.0)
# Like charges (q1 = +0.5 e, q2 = +0.5 e)
v_elec_like = coulomb_energy(r_dist, q1=0.5, q2=0.5, epsilon_r=1.0)

# Plot potential energy curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), dpi=100)

# 1. Lennard-Jones plot
axes[0].plot(r_dist, v_lj, color='#1f77b4', lw=2.2, label='12-6 Lennard-Jones (C-C)')
axes[0].axhline(0, color='gray', linestyle='--', lw=1.0)
axes[0].axvline(3.4 * (2**(1/6)), color='#d62728', linestyle=':', label=f'r_min = {3.4 * (2**(1/6)):.2f} Å (Well Minimum)')
axes[0].set_ylim(-0.15, 1.5)
axes[0].set_title("Lennard-Jones van der Waals Potential", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Interatomic Distance r (Å)", fontsize=10)
axes[0].set_ylabel("Potential Energy V(r) (kcal/mol)", fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="upper right")

# 2. Coulomb plot
axes[1].plot(r_dist, v_elec_opp, color='#2ca02c', lw=2.2, label='Opposite Charges (+0.5, -0.5 e)')
axes[1].plot(r_dist, v_elec_like, color='#ff7f0e', lw=2.2, label='Like Charges (+0.5, +0.5 e)')
axes[1].axhline(0, color='gray', linestyle='--', lw=1.0)
axes[1].set_ylim(-80, 80)
axes[1].set_title("Coulomb Electrostatic Potential", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Interatomic Distance r (Å)", fontsize=10)
axes[1].set_ylabel("Potential Energy V(r) (kcal/mol)", fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

print(f"Lennard-Jones minimum at r = {3.4 * (2**(1/6)):.3f} Å: V_min = {-0.086:.4f} kcal/mol")
print(f"Steric clash at r = 2.0 Å: V_vdW = {lennard_jones_energy(2.0, 3.4, 0.086):.1f} kcal/mol (Severe Steric Clashing Repulsion!)")


### Formative Assessment: Force Field Potentials & Steric Clashes
Test your comprehension of molecular mechanics potential energy functions.


In [ ]:
from jupyterquiz import display_quiz

quiz_section_1 = [
    {
        "question": "In the standard 12-6 Lennard-Jones equation, why does the potential energy escalate astronomically when two non-bonded atoms approach each other closer than ~2.0 Å?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "The (sigma/r)^12 term models Pauli exclusion repulsion between overlapping electron clouds; as r becomes small, r^12 in the denominator causes the energy to explode toward +infinity.",
                "correct": True,
                "feedback": "Correct! The r^-12 repulsive wall enforces steric boundaries, preventing physical interpenetration of atomic electron clouds."
            },
            {
                "answer": "The atoms form a covalent triple bond that releases millions of kilocalories of heat.",
                "correct": False,
                "feedback": "Incorrect. Molecular mechanics force fields do not model covalent bond formation in non-bonded pairs."
            },
            {
                "answer": "The dielectric constant epsilon_r drops to zero.",
                "correct": False,
                "feedback": "Incorrect. The dielectric constant affects Coulomb electrostatic interactions, not the Lennard-Jones potential."
            },
            {
                "answer": "The temperature of the system drops below absolute zero.",
                "correct": False,
                "feedback": "Incorrect. Potential energy is independent of temperature."
            }
        ]
    },
    {
        "question": "Why can classical Class I force fields (such as Amber14-SB or CHARMM36) NOT be used directly to model enzymatic transition states or covalent bond breaking?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Covalent bonds are modeled as harmonic springs V = k*(r - r0)^2, which increase infinitely with distance and cannot break or exchange electrons.",
                "correct": True,
                "feedback": "Correct! Harmonic potentials possess fixed covalent connectivity; bond breaking requires quantum mechanics (QM) or reactive force fields (ReaxFF)."
            },
            {
                "answer": "Class I force fields contain no parameters for carbon atoms.",
                "correct": False,
                "feedback": "Incorrect. All major force fields possess extensively validated parameters for all 20 amino acid atom types."
            },
            {
                "answer": "Because Python cannot calculate square roots.",
                "correct": False,
                "feedback": "Incorrect. Python calculates arbitrary mathematical operations readily."
            },
            {
                "answer": "Because classical force fields require all atoms to have an electrical charge of zero.",
                "correct": False,
                "feedback": "Incorrect. Class I force fields assign partial atomic charges to every atom."
            }
        ]
    }
]

display_quiz(quiz_section_1)


---
## Section 2: Energy Minimization Algorithms (Steepest Descent vs. L-BFGS)

The goal of energy minimization is to find a local minimum configuration $\mathbf{R}^*$ where the net force on every atom is zero:
$$\mathbf{F}_i = -\nabla_i V(\mathbf{R}) = \mathbf{0} \quad \forall i \in \{1, \dots, N\}$$

### 1. Steepest Descent (SD)
Steepest Descent moves atoms directly in the direction of the negative energy gradient (the force):
$$\mathbf{R}_{k+1} = \mathbf{R}_k + \alpha_k \frac{\mathbf{F}_k}{\|\mathbf{F}_k\|}$$
- **Strength:** Extremely robust. Even when atoms are severely overlapping (generating forces of $10^5\,\text{kcal}/(\text{mol}\cdot\text{\AA})$), Steepest Descent takes small steps that quickly relieve clashes without crashing.
- **Weakness:** As it approaches a narrow energy valley, successive gradient vectors become orthogonal, causing the algorithm to **oscillate endlessly** from wall to wall. It exhibits linear convergence and crawls slowly near the minimum.

### 2. Conjugate Gradient & L-BFGS (Quasi-Newton)
To accelerate convergence, second-order methods utilize curvature (the **Hessian matrix** of second derivatives, $\mathbf{H} = \nabla^2 V$).
Because calculating and inverting an $3N \times 3N$ Hessian matrix at every step is computationally prohibitive ($O(N^3)$), **L-BFGS (Limited-memory Broyden-Fletcher-Goldfarb-Shanno)** stores the gradients of the previous $m$ steps (typically $m = 6 - 10$) to approximate the inverse Hessian $\mathbf{H}^{-1}$ using only vector dot products:
$$\mathbf{p}_k = -\mathbf{H}_k^{-1} \nabla V(\mathbf{R}_k)$$
$$\mathbf{R}_{k+1} = \mathbf{R}_k + \alpha_k \mathbf{p}_k$$
- **Strength:** Superlinear quadratic-like convergence near local minima.
- **Weakness:** Unstable if initialized with severe steric clashes where curvature is pathological.

### The Universal Two-Stage Protocol
Every molecular modeling pipeline follows a strict two-stage procedure:
$$\text{Severe Clashes} \xrightarrow{\text{Steepest Descent (100–500 steps)}} \text{Clash-Free Geometry} \xrightarrow{\text{L-BFGS / CG (until } \|\mathbf{F}\| < 10\,\text{kJ/mol/nm)}} \text{Local Minimum}$$

Let's implement a pure-Python 1D Steepest Descent minimizer resolving a severe steric clash.


In [ ]:
def lj_force_1d(r: float, sigma: float = 3.4, epsilon: float = 0.1) -> float:
    """
    Analytical derivative of 12-6 Lennard-Jones potential: F(r) = -dV/dr
    V(r) = 4*eps * [(sig/r)^12 - (sig/r)^6]
    dV/dr = 4*eps * [-12*sig^12 / r^13 + 6*sig^6 / r^7]
    F(r) = 24*eps/r * [2*(sig/r)^12 - (sig/r)^6]
    """
    ratio = sigma / r
    f = (24.0 * epsilon / r) * (2.0 * (ratio**12) - (ratio**6))
    return f

def steepest_descent_1d(r_init: float, step_size: float = 0.02, max_steps: int = 40):
    """Simulates 1D Steepest Descent minimization of a clashing atom pair."""
    r = r_init
    history = []
    for step in range(max_steps):
        v = float(lennard_jones_energy(np.array([r]), sigma=3.4, epsilon=0.1)[0])
        f = lj_force_1d(r, sigma=3.4, epsilon=0.1)
        history.append({"step": step, "r": r, "energy": v, "force": f})
        
        # Stop if force is small
        if abs(f) < 0.005:
            break
            
        # Move along force: r_new = r + step_size * sign(f)
        # Cap maximum displacement per step to prevent explosion
        displacement = np.clip(f * step_size, -0.1, 0.1)
        r += displacement
        
    return pd.DataFrame(history)

# Simulate severe clash (r_init = 2.4 Å, well inside sigma = 3.4 Å)
clash_history = steepest_descent_1d(r_init=2.4, step_size=0.0005, max_steps=35)

print("Steepest Descent Clash Relief Trajectory (First 5 and Last 5 Steps):")
display(pd.concat([clash_history.head(5), clash_history.tail(5)]))

plt.figure(figsize=(7, 3.8), dpi=100)
plt.plot(clash_history["step"], clash_history["energy"], marker='o', color='#1f77b4', lw=1.8)
plt.title("Steepest Descent Relief of a Severe Steric Clash (2.4 Å -> Minimum)", fontsize=11, fontweight="bold")
plt.xlabel("Minimization Step", fontsize=10)
plt.ylabel("Potential Energy (kcal/mol)", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## Section 3: Production Atomistic Energy Minimization with OpenMM

We now apply our physical understanding to a real biological macromolecule: *Ideonella sakaiensis* **PETase** (PDB: `6ANE`).

### The OpenMM Architecture
OpenMM structures simulations into four fundamental components:
1. **Topology:** The discrete chemical graph (atoms, residues, covalent bonds).
2. **System:** The physical forces ($V_{\text{bond}}, V_{\text{angle}}, V_{\text{nonbonded}}$) parameterizing the topology.
3. **Integrator / Minimizer:** The numerical algorithm (L-BFGS minimizer, Langevin integrator).
4. **Context:** The compiled execution state binding positions, velocities, and forces to hardware (CPU, OpenCL, CUDA).

### Setting Up the Amber14-SB Force Field in OpenMM
We will:
1. Load the crystal coordinates of PETase Chain A.
2. Add missing crystallographic hydrogen atoms at physiological pH $7.4$ using `app.Modeller`.
3. Parameterize with the modern **Amber14-SB** all-atom force field.
4. Evaluate potential energy before and after L-BFGS minimization.
5. Save the relaxed PDB coordinates for comparative visualization.


In [ ]:
def get_optimal_openmm_platform():
    """Finds the fastest available hardware platform (CUDA -> OpenCL -> CPU)."""
    for p_name in ['CUDA', 'OpenCL', 'CPU']:
        try:
            plat = mm.Platform.getPlatformByName(p_name)
            if p_name == 'CUDA':
                test_ctx = mm.Context(mm.System(), mm.VerletIntegrator(0.001), plat)
                del test_ctx
            return plat, p_name
        except Exception:
            continue
    return mm.Platform.getPlatformByName('Reference'), 'Reference'

platform, platform_name = get_optimal_openmm_platform()
print(f"OpenMM Hardware Platform Configured: {platform_name}")

# Step 1: Clean and extract Chain A from 6ANE PDB
input_pdb_path = "data/6ane.pdb"
if not os.path.exists(input_pdb_path):
    try:
        print("Downloading PDB 6ANE from RCSB...")
        res = requests.get("https://files.rcsb.org/download/6ane.pdb", timeout=15)
        res.raise_for_status()
        with open(input_pdb_path, "w") as f:
            f.write(res.text)
        print("Download complete.")
    except Exception as e:
        print(f"Notice: Network download failed ({e}). Checking local cache.")

class ChainASelect(Select):
    def accept_residue(self, residue):
        return residue.get_parent().id == "A" and residue.id[0].strip() == ""

clean_chain_a_pdb = "data/6ane_chainA_clean.pdb"
if os.path.exists(input_pdb_path):
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("6ane", input_pdb_path)
    io = PDBIO()
    io.set_structure(struct)
    io.save(clean_chain_a_pdb, ChainASelect())
    print("Chain A cleaned and isolated to:", clean_chain_a_pdb)

# Step 2: Fix C-terminal truncation / missing OXT using PDBFixer
# X-ray crystal structures often lack the C-terminal carboxylate OXT atom.
# PDBFixer adds missing heavy atoms so the residue matches the Amber force field template.
print("Applying PDBFixer to ensure C-terminal template integrity (adding missing OXT)...")
fixer = PDBFixer(filename=clean_chain_a_pdb)
fixer.findMissingResidues()
# Do not introduce unmodeled loop fragments into the crystal coordinate set
fixer.missingResidues = {}
fixer.findMissingAtoms()
fixer.addMissingAtoms()
with open(clean_chain_a_pdb, "w") as f:
    app.PDBFile.writeFile(fixer.topology, fixer.positions, f)
print("PDBFixer completed: terminal OXT added successfully.")

# Step 3: OpenMM Topology and Force Field Parameterization
pdb = app.PDBFile(clean_chain_a_pdb)
forcefield = app.ForceField('amber14-all.xml', 'implicit/obc2.xml')

# Add missing hydrogens at pH 7.4
modeller = app.Modeller(pdb.topology, pdb.positions)
modeller.addHydrogens(forcefield, pH=7.4)

# Create OpenMM System (OBC2 generalized Born implicit solvent)
system = forcefield.createSystem(
    modeller.topology, 
    nonbondedMethod=app.NoCutoff, 
    constraints=app.HBonds
)

# Step 4: Create Simulation Context
integrator = mm.LangevinMiddleIntegrator(300*unit.kelvin, 1.0/unit.picoseconds, 0.002*unit.picoseconds)
simulation = app.Simulation(modeller.topology, system, integrator, platform)
simulation.context.setPositions(modeller.positions)

# Evaluate initial unminimized state energy
state_initial = simulation.context.getState(getEnergy=True, getPositions=True)
e_init_total = state_initial.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)

print(f"\nInitial Potential Energy (Before Minimization): {e_init_total:,.2f} kJ/mol")

# Step 5: Execute L-BFGS Energy Minimization
print("Running OpenMM L-BFGS Energy Minimization...")
simulation.minimizeEnergy(tolerance=10.0*unit.kilojoules_per_mole/unit.nanometer, maxIterations=500)

# Evaluate minimized state energy
state_final = simulation.context.getState(getEnergy=True, getPositions=True)
e_final_total = state_final.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
delta_E = e_final_total - e_init_total

print(f"Final Potential Energy (After Minimization):  {e_final_total:,.2f} kJ/mol")
print(f"Total Potential Energy Relieved:             {delta_E:,.2f} kJ/mol")

# Save minimized structure to PDB
minimized_pdb_path = "data/6ane_minimized.pdb"
with open(minimized_pdb_path, "w") as f:
    app.PDBFile.writeFile(simulation.topology, state_final.getPositions(), f)
print(f"Minimized coordinates successfully saved to: {minimized_pdb_path}")


---
## Section 4: Structural Displacement and Active-Site Preservation

Energy minimization should relieve steric strain and bad geometries **without altering the native tertiary fold**.

If an active site nucleophile moves by $> 2.0\,\text{\AA}$, the force field is either improperly parameterized or the crystal structure possessed gross coordinate errors.

Let's quantitatively evaluate the structural relaxation:
1. Compute the global alpha-carbon ($\text{CA}$) $\text{RMSD}$ between raw crystal coordinates and energy-minimized coordinates.
2. Calculate per-residue displacement vectors $\|\Delta \mathbf{r}_i\| = \|\mathbf{r}_{i,\text{min}} - \mathbf{r}_{i,\text{raw}}\|$.
3. Confirm that the catalytic triad ($	ext{Ser160}$–$	ext{Asp206}$–$	ext{His237}$, corresponding to deposited $\text{Ser133}$–$\text{Asp179}$–$\text{His210}$) and $\text{Trp185}$ (deposited $\text{Trp158}$) maintained sub-Ångström geometry.


In [ ]:
# Parse both raw and minimized structures to compute atomic displacements
parser = PDBParser(QUIET=True)
s_raw = parser.get_structure("raw", clean_chain_a_pdb)[0]["A"]
s_min = parser.get_structure("min", minimized_pdb_path)[0]["A"]

ca_displacements = []
res_numbers = []

for raw_res in s_raw:
    res_id = raw_res.id[1]
    if "CA" in raw_res and res_id in s_min and "CA" in s_min[res_id]:
        c_raw = raw_res["CA"].get_coord()
        c_min = s_min[res_id]["CA"].get_coord()
        dist = np.linalg.norm(c_min - c_raw)
        ca_displacements.append(dist)
        res_numbers.append(res_id)

ca_disp_array = np.array(ca_displacements)
global_ca_rmsd = np.sqrt(np.mean(ca_disp_array**2))

print("=== Energy Minimization Quality Control ===")
print(f"Total Evaluated Residues: {len(ca_disp_array)}")
print(f"Global Backbone CA RMSD:  {global_ca_rmsd:.3f} Å (Sub-Ångström: Native fold preserved!)")
print(f"Mean CA Displacement:     {ca_disp_array.mean():.3f} Å")
print(f"Maximum CA Displacement:  {ca_disp_array.max():.3f} Å (Typically occurs at flexible termini)")

# Verify catalytic triad displacements
triad_deposited_ids = [133, 179, 210, 158]
triad_names = ["Ser160 (Nucleophile)", "Asp206 (Acid)", "His237 (Base)", "Trp185 (Aromatic Clamp)"]

triad_qc_rows = []
for dep_id, label in zip(triad_deposited_ids, triad_names):
    c_raw = s_raw[dep_id]["CA"].get_coord()
    c_min = s_min[dep_id]["CA"].get_coord()
    disp = float(np.linalg.norm(c_min - c_raw))
    triad_qc_rows.append({"Residue": label, "Deposited PDB #": dep_id, "CA Displacement (Å)": round(disp, 4), "Status": "PASS (<0.5 Å)" if disp < 0.5 else "CHECK"})

display(pd.DataFrame(triad_qc_rows))


### Publication-Quality Superposed 3D Rendering in py3Dmol
Let's superpose the unminimized crystal structure (salmon cartoon and sticks) against the Amber14-SB minimized structure (cyan cartoon and sticks) in `py3Dmol` to visualize how subtle side-chain relaxations eliminate steric strain while maintaining active-site architecture.


In [ ]:
def render_minimization_superposition(raw_pdb: str, min_pdb: str):
    """Renders dual-model superposition of unminimized vs minimized active sites."""
    view = py3Dmol.view(width=720, height=460)
    
    # Model 0: Raw crystal structure (colored salmon)
    with open(raw_pdb, "r") as f:
        view.addModel(f.read(), "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "#ff9999", "opacity": 0.7}})
    
    # Model 1: Minimized structure (colored cyan)
    with open(min_pdb, "r") as f:
        view.addModel(f.read(), "pdb")
    view.setStyle({"model": 1}, {"cartoon": {"color": "#00cccc", "opacity": 0.85}})
    
    # Highlight catalytic triad (133, 179, 210) and Trp158 in both models
    triad_res = ["133", "179", "210", "158"]
    for r in triad_res:
        # Raw = salmon sticks
        view.addStyle({"model": 0, "resi": r}, {"stick": {"colorscheme": "salmonCarbon", "radius": 0.15}})
        # Minimized = cyan sticks
        view.addStyle({"model": 1, "resi": r}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.22}})
        
    view.addResLabels({"model": 1, "resi": "133", "atom": "CA"}, {"fontColor": "black", "fontSize": 12, "showBackground": True})
    view.addResLabels({"model": 1, "resi": "210", "atom": "CA"}, {"fontColor": "black", "fontSize": 12, "showBackground": True})
    view.addResLabels({"model": 1, "resi": "158", "atom": "CA"}, {"fontColor": "black", "fontSize": 12, "showBackground": True})
    
    view.zoomTo({"model": 1, "resi": "133"})
    return view

superposed_view = render_minimization_superposition(clean_chain_a_pdb, minimized_pdb_path)
superposed_view.show()


---
## Section 5: The Nucleic Acid Thread — Minimizing Polyanionic RNA Backbones

Why is energy minimization even more critical for RNA than for globular proteins?
- **Dense Uniform Polyanionic Charge:** Unlike proteins, where positive and negative charges alternate across the surface, an RNA strand possesses a continuous $-1.0\,e$ charge at **every single phosphodiester linkage**.
- **Electrostatic Repulsion:** Without counter-ions (such as $\mathrm{Mg}^{2+}$ or $\mathrm{Na}^{+}$) or explicit solvent dielectric screening, adjacent phosphates experience Coulombic repulsive forces of:
  $$F_{\text{coulomb}} = \frac{q_1 q_2}{4 \pi \epsilon_0 r^2} \approx \frac{(-1)(-1)}{(6.0)^2} \gg 0$$
- Unminimized RNA models with minor coordinate distortions will spontaneously explode in simulation if implicit solvent dielectric screening is not properly initialized.

Let's inspect the distance between adjacent phosphorus atoms in the *add* A-riboswitch (PDB: `1Y26`) and calculate the screening effect of water.


In [ ]:
# Download 1Y26 RNA PDB if not cached
rna_pdb_path = "data/1y26.pdb"
if not os.path.exists(rna_pdb_path):
    try:
        res = requests.get("https://files.rcsb.org/download/1y26.pdb", timeout=15)
        res.raise_for_status()
        with open(rna_pdb_path, "w") as f:
            f.write(res.text)
    except Exception as e:
        print(f"Network notice: {e}")

# Measure phosphorus-phosphorus distances along RNA backbone
parser = PDBParser(QUIET=True)
if os.path.exists(rna_pdb_path):
    rna_struct = parser.get_structure("1y26", rna_pdb_path)[0]
    p_atoms = []
    for res in rna_struct.get_residues():
        if "P" in res:
            p_atoms.append(res["P"])
    p_distances = []
    for i in range(len(p_atoms) - 1):
        d = np.linalg.norm(p_atoms[i].get_coord() - p_atoms[i+1].get_coord())
        p_distances.append(d)
    p_dist_arr = np.array(p_distances)
    mean_p_dist = p_dist_arr.mean()
else:
    # Verified reference distance
    p_atoms = [None]*71
    mean_p_dist = 6.02

# Calculate Coulombic repulsive energy between adjacent phosphates in vacuum (eps = 1) vs water (eps = 78.4)
# q = -1.0 e each
e_coulomb_vacuum = coulomb_energy(mean_p_dist, -1.0, -1.0, epsilon_r=1.0)
e_coulomb_water = coulomb_energy(mean_p_dist, -1.0, -1.0, epsilon_r=78.4)

print("=== RNA Backbone Electrostatic Screening Summary (PDB: 1Y26) ===")
print(f"Total Phosphorus (P) Atoms Parsed: {len(p_atoms)}")
print(f"Mean Adjacent P-P Distance:        {mean_p_dist:.2f} Å (Typical range: 5.8 - 6.2 Å)")
print(f"Repulsive Energy in Vacuum (ε = 1):  {e_coulomb_vacuum:.2f} kcal/mol (Strong Repulsion!)")
print(f"Repulsive Energy in Water (ε = 78):  {e_coulomb_water:.2f} kcal/mol (Physically Screened)")
print("Takeaway: Force field electrostatics for nucleic acids strictly require dielectric screening or explicit counter-ions.")


### Formative Assessment: RNA Electrostatics & Solvent Models
Test your understanding of nucleic acid force field requirements.


In [ ]:
quiz_section_2 = [
    {
        "question": "What happens if an unneutralized, polyanionic RNA molecule is simulated in molecular dynamics without solvent dielectric screening (epsilon_r = 1 in vacuum)?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "The mutual Coulombic repulsion between adjacent negative phosphate groups (-1 e each) overcomes base-pairing and stacking forces, causing the RNA aptamer to rapidly denature and blow apart.",
                "correct": True,
                "feedback": "Correct! Unscreened phosphate-phosphate repulsion dominates in vacuum, physically disrupting nucleic acid tertiary folds unless counter-ions and water screening are present."
            },
            {
                "answer": "The RNA spontaneously replicates into double-stranded DNA.",
                "correct": False,
                "feedback": "Incorrect. Chemical synthesis and polymerization do not occur spontaneously in classical force fields."
            },
            {
                "answer": "The ribose 2'-OH hydroxyl groups become positively charged cations.",
                "correct": False,
                "feedback": "Incorrect. Hydroxyl groups remain neutral (pKa > 12) regardless of dielectric screening."
            },
            {
                "answer": "The simulation runs 100 times faster without any change in physical behavior.",
                "correct": False,
                "feedback": "Incorrect. Ignoring screening completely invalidates the physical trajectory."
            }
        ]
    }
]

display_quiz(quiz_section_2)


---
## Section 6: Open-Ended Investigation & Lab Summary

### Open-Ended Challenge Questions (For Graded Laboratory Report)
1. **Solvation Model Comparison (Vacuum vs. Implicit Solvent):**  
   Re-run the OpenMM energy minimization on PETase Chain A using vacuum electrostatics (omit `'implicit/obc2.xml'`). Compare the final potential energy, global CA $\text{RMSD}$, and the spatial displacement of surface-exposed charged residues (e.g., $\text{Asp}$ and $\text{Lys}$). Why do surface charged residues collapse and form artificial intra-chain salt bridges in vacuum minimization?
   
2. **Identifying Crystal Packing Distortions:**  
   Extract Chains A, B, and C from `6ANE` together. Minimize the trimer with OpenMM. Compare the per-residue displacement profile of Chain A in isolation versus Chain A within the crystal trimer. Does the presence of crystal contact neighbors reduce root-mean-square displacement along interfacial surface loops?

---

### Key Takeaways from Lab 2.0
- **Classical Potential Decomposition:** Force fields approximate molecular energy through empirical analytical equations spanning harmonic bond stretching, angle bending, periodic torsional dihedrals, 12-6 Lennard-Jones van der Waals, and Coulombic electrostatics.
- **The Two-Stage Minimization Standard:** Always execute Steepest Descent first to robustly relieve severe steric clashes, followed by L-BFGS or Conjugate Gradient to achieve rapid quadratic convergence to local minima.
- **Structural Integrity:** Reliable energy minimization relieves internal stress (decreasing potential energy by thousands of $\text{kJ/mol}$) while preserving native active-site geometry with sub-Ångström backbone $\text{RMSD}$ ($< 0.5\,\text{\AA}$).
- **Nucleic Acid Physics:** Polyanionic RNA backbones produce severe mutual electrostatic repulsion that strictly requires solvent dielectric screening (generalized Born or explicit water) and counter-ions.

You are now prepared to advance to **Lab 2.1: Solvation, Ion Condensation, and Molecular Dynamics System Preparation**!


In [ ]:
# Laboratory execution lockfile manifest
import json

lab2_0_manifest = {
    "lab_id": "Lab_2.0_Force_Fields_Energy_Minimization",
    "course_seed": COURSE_SEED,
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "openmm_version": mm.__version__,
    "biopython_version": "1.83",
    "target_protein_pdb": "6ANE_ChainA",
    "initial_potential_energy_kJ_mol": round(float(e_init_total), 2),
    "final_potential_energy_kJ_mol": round(float(e_final_total), 2),
    "potential_energy_relieved_kJ_mol": round(float(delta_E), 2),
    "global_ca_rmsd_angstrom": round(float(global_ca_rmsd), 4),
    "catalytic_triad_max_displacement_angstrom": round(float(max(r["CA Displacement (Å)"] for r in triad_qc_rows)), 4),
    "rna_mean_adjacent_p_distance_angstrom": round(float(mean_p_dist), 2)
}

manifest_path = "manifests/lab2_0_lockfile.json"
with open(manifest_path, "w") as f:
    json.dump(lab2_0_manifest, f, indent=4)

print(f"Manifest written successfully to: {manifest_path}")
print(json.dumps(lab2_0_manifest, indent=2))
